In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import os
import torch
import pickle
import config

from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder

from src.metric import model_evaluation 
from src.datasets import ResumeJDDataset
from src.cross_encoder_training import compute_batch_scores


In [ ]:
path=config.CLEANED_DATA_DIR

In [ ]:
with open(os.path.join(path,'train_df.pkl'),'rb') as f:
    train_df=pickle.load(f)
    
with open(os.path.join(path,'val_df.pkl'),'rb') as f:
    val_df=pickle.load(f)
        
with open(os.path.join(path,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)
    


In [ ]:
BATCH_SIZE=config.CHUNK_BATCH_SIZE

In [ ]:
cross_encoder= CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',num_labels=1,device=config.device)

In [ ]:
labels=[float(config.label_to_score[label]) for label in train_df['label']]
train_dataset=ResumeJDDataset(train_df['resume_text'].values,train_df['job_description_text'].values,labels)

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [ ]:
min_delta=0.01
count=0
best_score=float('-inf')
epochs=3
patience=2

In [ ]:
best_model_path=os.path.join(config.CHUNKED_MODEL_DIR,'cross_encoder_chunks')
os.makedirs(config.CHUNKED_MODEL_DIR,exist_ok=True)

In [ ]:
optimizer = torch.optim.AdamW(cross_encoder.parameters(), lr=2e-5)
loss_fn = torch.nn.MSELoss()

In [ ]:
for epoch in range(epochs):
    
    print(f"Epoch {epoch+1}/{epochs}")
    cross_encoder.train()
    
    total_loss=0
    
    progress_bar = tqdm(train_loader, desc="Training")
    
    for resumes,jds,labels in progress_bar:
        optimizer.zero_grad()
        
        scores=compute_batch_scores(cross_encoder,resumes,jds)
        labels=labels.to(config.device)
        
        loss=loss_fn(scores,labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss+=loss.item()
        
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")
        
    average_loss=total_loss/len(progress_bar)
    print("Average Loss:",average_loss)
        
        
    #Validation    
    cross_encoder.eval()
    
    with torch.no_grad():
            
        val_scores=compute_batch_scores(cross_encoder,val_df['resume_text'].values,val_df['job_description_text'].values)
        val_scores=val_scores.cpu().numpy()
       
        metrics=model_evaluation(val_scores,val_df,"job_description_text")
        print("NDCG:", metrics["ndcg_val"])
        print("MAP:", metrics["map_score"])
        
        final_score=0.6*metrics["ndcg_val"]+0.3*metrics["map_score"]+0.1*metrics["mrr_score"]
        
    if final_score>best_score+min_delta:
        best_score=final_score
        cross_encoder.save(best_model_path)
        count=0
    else:
        count+=1

    if count==patience:
        print("Early stopping triggered.")
        break
                
            
    

In [ ]:
cross_encoder= CrossEncoder(best_model_path,num_labels=1,device=config.device)

In [ ]:
cross_encoder.eval()
with torch.no_grad():
    scores=compute_batch_scores(cross_encoder,test_df['resume_text'].values,test_df['job_description_text'].values)
    scores=scores.cpu().numpy()
    metrics=model_evaluation(scores,test_df,"job_description_text")

In [ ]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])